<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Kitchen Appliance IoT Logs - Feature Engineering</title>
    <style>
        body { font-family: "Segoe UI", Arial, sans-serif; background:black; padding: 30px; line-height:1.8;}
        h1, h2, h3 { color: #1b2b4a; }
        .section { background:black; padding:25px; margin-bottom:30px; border-radius:10px; box-shadow:0 0 10px rgba(0,0,0,0.05);}
        code { display:block; background:#1e1e1e; color:#eaeaea; padding:14px; border-radius:6px; margin-top:12px; font-size:14px; overflow-x:auto;}
        .note { background:#e3f2fd; padding:12px; border-left:5px solid #2196f3; margin-top:12px;}
        .warning { background:#fdecea; padding:12px; border-left:5px solid #e53935; margin-top:12px;}
        .example { background:#f1f8e9; padding:12px; border-left:5px solid #7cb342; margin-top:12px;}
        table { border-collapse: collapse; width:100%; margin-top:12px;}
        th, td { border:1px solid #ccc; padding:10px; text-align:left; }
        th { background:#f0f0f0; }
    </style>
</head>
<body>

<h1>📊 Feature Engineering for Kitchen Appliance IoT Logs</h1>
<p>Complete theory, math, examples, and Python code for creating new columns from IoT sensor logs.</p>

<!-- SECTION 1 -->
<div class="section">
<h2>1️⃣ Timestamp Features</h2>
<p><strong>Theory:</strong> Extracting features from timestamp helps detect usage patterns across hours, days, weeks, or months. Many appliances have peak usage times (daily or weekly).</p>
<p><strong>Math:</strong> None directly; these are categorical or cyclic features derived from datetime.</p>

<table>
<tr><th>Column</th><th>Purpose</th><th>Example</th><th>Python Code</th></tr>
<tr>
<td>hour</td>
<td>Detect daily usage pattern</td>
<td>10 → 10 AM</td>
<td><code>df['hour'] = df.index.hour</code></td>
</tr>
<tr>
<td>day_of_week</td>
<td>Detect weekday/weekend patterns</td>
<td>0 → Monday</td>
<td><code>df['day_of_week'] = df.index.dayofweek</code></td>
</tr>
<tr>
<td>is_weekend</td>
<td>Binary flag for weekend usage</td>
<td>1 → Saturday/Sunday</td>
<td><code>df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)</code></td>
</tr>
<tr>
<td>month</td>
<td>Detect seasonal appliance usage</td>
<td>1 → January</td>
<td><code>df['month'] = df.index.month</code></td>
</tr>
<tr>
<td>day</td>
<td>Detect day-of-month patterns</td>
<td>15 → 15th day</td>
<td><code>df['day'] = df.index.day</code></td>
</tr>
<tr>
<td>week</td>
<td>Weekly aggregation patterns</td>
<td>3 → 3rd week</td>
<td><code>df['week'] = df.index.isocalendar().week</code></td>
</tr>
</table>

<div class="example">Example: Oven usage peaks at 7–9 PM (hour), Dishwasher usage peaks on weekends (is_weekend).</div>
</div>

<!-- SECTION 2 -->
<div class="section">
<h2>2️⃣ Device Usage Features</h2>
<p><strong>Theory:</strong> Capture real usage behavior, duration, energy consumption, and appliance status. Useful for maintenance and customer usage analysis.</p>
<p><strong>Math:</strong></p>
<ul>
<li><strong>Usage duration:</strong> difference between consecutive timestamps <code>duration = t_i - t_{i-1}</code></li>
<li><strong>Power consumption:</strong> energy = power × duration <code>kWh = P × t</code></li>
<li>Status and error flags are encoded numerically for modeling.</li>
</ul>

<table>
<tr><th>Column</th><th>Purpose</th><th>Example</th><th>Python Code</th></tr>
<tr>
<td>usage_duration</td>
<td>Time appliance is ON</td>
<td>60 seconds</td>
<td><code>df['usage_duration'] = df.index.to_series().diff().dt.seconds.fillna(0)</code></td>
</tr>
<tr>
<td>power_consumption</td>
<td>Energy consumed per interval</td>
<td>180 W × 60 sec = 3 Wh</td>
<td><code>df['power_consumption'] = df['power'] * df['usage_duration']</code></td>
</tr>
<tr>
<td>status_flag</td>
<td>Encode categorical status numerically</td>
<td>RUNNING → 2</td>
<td><code>df['status_flag'] = df['status'].map({'OFF':0,'IDLE':1,'RUNNING':2})</code></td>
</tr>
<tr>
<td>error_flag</td>
<td>Binary flag for errors</td>
<td>1 if error_code not null</td>
<td><code>df['error_flag'] = df['error_code'].notna().astype(int)</code></td>
</tr>
</table>

<div class="example">Example: Coffee machine short bursts → usage_duration identifies real brewing events. Oven logs “RUNNING” → status_flag=2.</div>
</div>

<!-- SECTION 3 -->
<div class="section">
<h2>3️⃣ Rolling / Aggregation Features</h2>
<p><strong>Theory:</strong> Time-series logs often have noise. Rolling averages smooth readings. Differences detect trends or sudden spikes.</p>
<p><strong>Math:</strong></p>
<ul>
<li>Moving average (MA): <code>MA_t = (x_t + x_{t-1} + ... + x_{t-k+1}) / k</code></li>
<li>Difference: <code>diff_t = x_t - x_{t-1}</code></li>
</ul>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>temp_MA_5</td>
<td>5-reading moving average of temperature</td>
<td><code>df['temp_MA_5'] = df['temperature'].rolling(5).mean()</code></td>
</tr>
<tr>
<td>power_MA_10</td>
<td>10-reading moving average of power</td>
<td><code>df['power_MA_10'] = df['power'].rolling(10).mean()</code></td>
</tr>
<tr>
<td>temp_diff</td>
<td>Change from previous temperature reading</td>
<td><code>df['temp_diff'] = df['temperature'].diff()</code></td>
</tr>
<tr>
<td>power_diff</td>
<td>Change in power reading</td>
<td><code>df['power_diff'] = df['power'].diff()</code></td>
</tr>
</table>

<div class="example">Example: Oven temperature fluctuates ±5°C → temp_MA_5 smooths trend. Power spike → power_diff highlights anomaly.</div>
</div>

<!-- SECTION 4 -->
<div class="section">
<h2>4️⃣ Anomaly / Outlier Features</h2>
<p><strong>Theory:</strong> Detect abnormal appliance behavior using statistics. Z-score and IQR methods are commonly used.</p>
<p><strong>Math:</strong></p>
<ul>
<li>Z-score: <code>z_i = (x_i - μ)/σ</code>, flag if |z_i|>3</li>
<li>IQR: <code>Q1 - 1.5*IQR &lt; x_i &gt; Q3 + 1.5*IQR</code></li>
</ul>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>temp_zscore</td>
<td>Detect temperature spikes</td>
<td><code>df['temp_zscore'] = (df['temperature'] - df['temperature'].mean())/df['temperature'].std()</code></td>
</tr>
<tr>
<td>power_zscore</td>
<td>Detect unusual power consumption</td>
<td><code>df['power_zscore'] = (df['power'] - df['power'].mean())/df['power'].std()</code></td>
</tr>
<tr>
<td>is_temp_outlier</td>
<td>Binary flag for temp outliers</td>
<td><code>df['is_temp_outlier'] = (df['temp_zscore'].abs() > 3).astype(int)</code></td>
</tr>
</table>

<div class="example">Example: Oven temperature jumps from 180°C → 250°C → flagged as outlier.</div>
</div>

<!-- SECTION 5 -->
<div class="section">
<h2>5️⃣ Lag / Cumulative Features</h2>
<p><strong>Theory:</strong> Capture trends over time and cumulative usage.</p>
<p><strong>Math:</strong></p>
<ul>
<li>Lag: <code>lag_t = x_t - x_{t-1}</code></li>
<li>Cumulative sum: <code>cumsum_t = Σ x_i</code></li>
<li>Rolling max: <code>max_t = max(x_t, x_{t-1}, ..., x_{t-k+1})</code></li>
</ul>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>temp_lag1</td>
<td>Previous reading of temperature</td>
<td><code>df['temp_lag1'] = df['temperature'].shift(1)</code></td>
</tr>
<tr>
<td>power_cumsum</td>
<td>Cumulative power usage</td>
<td><code>df['power_cumsum'] = df['power'].cumsum()</code></td>
</tr>
<tr>
<td>rolling_max_power</td>
<td>Max power over last 10 readings</td>
<td><code>df['rolling_max_power'] = df['power'].rolling(10).max()</code></td>
</tr>
</table>
</div>

<!-- SECTION 6 -->
<div class="section">
<h2>6️⃣ Duplicate / Missing Value Features</h2>
<p><strong>Theory:</strong> Flag quality issues in logs for data cleaning or modeling.</p>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>is_duplicate</td>
<td>Row is a duplicate</td>
<td><code>df['is_duplicate'] = df.duplicated().astype(int)</code></td>
</tr>
<tr>
<td>missing_flag_temp</td>
<td>Temperature missing value</td>
<td><code>df['missing_flag_temp'] = df['temperature'].isna().astype(int)</code></td>
</tr>
</table>

<div class="note">Duplicate spikes or missing values often indicate hardware or network issues.</div>
</div>

<!-- SECTION 7 -->
<div class="section">
<h2>✅ Key Takeaways</h2>
<ul>
<li>Timestamp features capture usage trends (hour, day, weekend, week).</li>
<li>Device usage features quantify appliance behavior (duration, energy, status).</li>
<li>Rolling and difference features smooth logs and detect sudden changes.</li>
<li>Anomaly/outlier features identify abnormal behavior using Z-score or IQR.</li>
<li>Lag and cumulative features capture trends over time.</li>
<li>Data quality flags (`is_duplicate`, `missing_flag`) ensure reliable analysis.</li>
<li>Feature engineering transforms raw IoT logs into actionable insights for analytics, maintenance, and customer usage behavior.</li>
</ul>
</div>

</body>
</html>
